# 03 — Weather Data Preparation and Load–Weather Integration

## Purpose

Prepare hourly weather covariates for the eight ERCOT weather zones and align them with ERCOT hourly electricity demand. The notebook:

1. Defines one representative weather location for each ERCOT zone.
2. Retrieves hourly historical weather observations for 2019–2025 using the Open-Meteo archive API and **ERA5** reanalysis.
3. Converts all timestamps to UTC and checks missing data, duplicated keys, and hourly continuity.
4. Validates that the weather time coverage spans the ERCOT load timestamps, including the early-2026 UTC boundary.
5. Reshapes regional load data, joins it with weather data, and exports quality-check summaries and a modeling-ready Parquet dataset.

**Scope and reproducibility.** Weather reanalysis values are proxies for each zone based on one representative point, not measurements at every customer site. Running the notebook requires access to the Open-Meteo archive endpoint and the cleaned load dataset produced earlier in this project. Tables below are previously saved results and have not been recomputed for this edited copy.



### 1. Imports and Display Settings

Import Python standard libraries and pandas, and configure readable table display.


In [ ]:
# Import core libraries and configure pandas display options.
from pathlib import Path

import json

import time

from urllib.parse import urlencode

from urllib.request import urlopen

import pandas as pd

pd.set_option(
    "display.max_columns",
    100,
)

pd.set_option(
    "display.float_format",
    lambda value: f"{value:,.3f}",
)


### 2. Project Paths

Resolve the repository root and create directories for raw and processed weather data. Printed paths are kept relative to the project to avoid exposing local user directories.


In [ ]:
# Locate the project root and prepare weather data directories.
current_directory = Path.cwd().resolve()

if (current_directory / "data").exists():
    project_root = current_directory
else:
    project_root = current_directory.parent

weather_data_dir = (
    project_root
    / "data"
    / "weather"
)

raw_weather_dir = (
    weather_data_dir
    / "raw"
)

processed_weather_dir = (
    weather_data_dir
    / "processed"
)

raw_weather_dir.mkdir(
    parents=True,
    exist_ok=True,
)

processed_weather_dir.mkdir(
    parents=True,
    exist_ok=True,
)

print("Project root: . (repository root)")

print("Raw weather directory:", raw_weather_dir.relative_to(project_root))

print("Processed weather directory:", processed_weather_dir.relative_to(project_root))


Project root: .
Raw weather directory: data/weather/raw
Processed weather directory: data/weather/processed


### 3. Representative Weather Locations

Map each ERCOT weather zone to one representative city and latitude/longitude pair. These points approximate zone-level weather exposure.


In [ ]:
# Define and validate representative ERCOT weather-zone locations.
weather_locations = pd.DataFrame([
    {
        "zone": "COAST",
        "location_name": "Houston",
        "latitude": 29.7604,
        "longitude": -95.3698,
    },
    {
        "zone": "EAST",
        "location_name": "Tyler",
        "latitude": 32.3513,
        "longitude": -95.3011,
    },
    {
        "zone": "FWEST",
        "location_name": "Midland",
        "latitude": 31.9973,
        "longitude": -102.0779,
    },
    {
        "zone": "NORTH",
        "location_name": "Wichita Falls",
        "latitude": 33.9137,
        "longitude": -98.4934,
    },
    {
        "zone": "NCENT",
        "location_name": "Dallas",
        "latitude": 32.7767,
        "longitude": -96.7970,
    },
    {
        "zone": "SOUTH",
        "location_name": "McAllen",
        "latitude": 26.2034,
        "longitude": -98.2300,
    },
    {
        "zone": "SCENT",
        "location_name": "San Antonio",
        "latitude": 29.4241,
        "longitude": -98.4936,
    },
    {
        "zone": "WEST",
        "location_name": "Abilene",
        "latitude": 32.4487,
        "longitude": -99.7331,
    },
])

if weather_locations["zone"].duplicated().any():
    raise ValueError(
        "Duplicate weather zones found."
    )

if not weather_locations["latitude"].between(-90, 90).all():
    raise ValueError(
        "Invalid latitude found."
    )

if not weather_locations["longitude"].between(-180, 180).all():
    raise ValueError(
        "Invalid longitude found."
    )

weather_locations_path = (
    weather_data_dir
    / "ercot_weather_zone_locations.csv"
)

weather_locations.to_csv(
    weather_locations_path.relative_to(project_root),
    index=False,
)

display(weather_locations)

print(
    "Weather locations saved to:",
    weather_locations_path.relative_to(project_root),
)


,zone,location_name,latitude,longitude
0,COAST,Houston,29.760,-95.370
1,EAST,Tyler,32.351,-95.301
2,FWEST,Midland,31.997,-102.078
3,NORTH,Wichita Falls,33.914,-98.493
4,NCENT,Dallas,32.777,-96.797
5,SOUTH,McAllen,26.203,-98.230
6,SCENT,San Antonio,29.424,-98.494
7,WEST,Abilene,32.449,-99.733


Weather locations saved to: data/weather/ercot_weather_zone_locations.csv


### 4. Weather Archive Parameters

Request temperature, apparent temperature, relative humidity, dew point, precipitation, and wind speed from the Open-Meteo historical archive.


In [ ]:
# Configure the Open-Meteo archive endpoint and hourly variables.
OPEN_METEO_ARCHIVE_URL = (
    "https://archive-api.open-meteo.com/v1/archive"
)

WEATHER_VARIABLES = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "wind_speed_10m",
]

print("Historical weather API:", OPEN_METEO_ARCHIVE_URL)

print("Hourly weather variables:", WEATHER_VARIABLES)


Historical weather API: https://archive-api.open-meteo.com/v1/archive
Hourly weather variables: ['temperature_2m', 'apparent_temperature', 'relative_humidity_2m', 'dew_point_2m', 'precipitation', 'wind_speed_10m']


### 5. ERA5 Download Helper

Retrieve hourly weather for one zone and requested interval. The request explicitly selects ERA5, uses UTC/GMT, and records both requested and returned grid coordinates.


In [ ]:
# Download an ERA5 hourly weather frame for one representative location.
def download_zone_weather(
    zone,
    location_name,
    latitude,
    longitude,
    start_date,
    end_date,
):
    request_parameters = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(WEATHER_VARIABLES),
         "models": "era5",
        "timezone": "GMT",
        "temperature_unit": "celsius",
        "wind_speed_unit": "ms",
        "precipitation_unit": "mm",
    }

    query_string = urlencode(
        request_parameters
    )

    request_url = (
        f"{OPEN_METEO_ARCHIVE_URL}"
        f"?{query_string}"
    )

    print(
        f"Downloading {zone} / {location_name}: "
        f"{start_date} to {end_date}"
    )

    with urlopen(
        request_url,
        timeout=120,
    ) as response:
        weather_response = json.load(response)

    if "hourly" not in weather_response:
        raise ValueError(
            f"No hourly weather data returned for {zone}."
        )

    weather_frame = pd.DataFrame(
        weather_response["hourly"]
    )

    weather_frame["weather_time_utc"] = pd.to_datetime(
        weather_frame["time"],
        utc=True,
        errors="raise",
    )

    weather_frame = weather_frame.drop(
        columns=["time"]
    )

    weather_frame.insert(
        0,
        "weather_time_utc",
        weather_frame.pop("weather_time_utc"),
    )

    weather_frame.insert(
        0,
        "zone",
        zone,
    )

    weather_frame.insert(
        1,
        "location_name",
        location_name,
    )

    weather_frame["requested_latitude"] = latitude

    weather_frame["requested_longitude"] = longitude

    weather_frame["grid_latitude"] = weather_response.get(
        "latitude"
    )

    weather_frame["grid_longitude"] = weather_response.get(
        "longitude"
    )

    weather_frame["grid_elevation_m"] = weather_response.get(
        "elevation"
    )

    return weather_frame


### 6. Single-Zone Sanity Check

Test Houston (COAST) for 2019, confirming the timestamp range and expected hourly observation count.


In [ ]:
# Test a single COAST/Houston calendar year.
coast_location = weather_locations.loc[
    weather_locations["zone"] == "COAST"
].iloc[0]

coast_2019_weather = download_zone_weather(
    zone=coast_location["zone"],
    location_name=coast_location["location_name"],
    latitude=coast_location["latitude"],
    longitude=coast_location["longitude"],
    start_date="2019-01-01",
    end_date="2019-12-31",
)

print(
    "COAST 2019 shape:",
    coast_2019_weather.shape,
)

print(
    "UTC start:",
    coast_2019_weather["weather_time_utc"].min(),
)

print(
    "UTC end:",
    coast_2019_weather["weather_time_utc"].max(),
)

display(
    coast_2019_weather.head(5)
)


COAST 2019 shape: (8760, 14)
UTC start: 2019-01-01 00:00:00+00:00
UTC end: 2019-12-31 23:00:00+00:00


,zone,location_name,weather_time_utc,temperature_2m,apparent_temperature,relative_humidity_2m,dew_point_2m,precipitation,wind_speed_10m,requested_latitude,requested_longitude,grid_latitude,grid_longitude,grid_elevation_m
0,COAST,Houston,2019-01-01 00:00:00+00:00,12.300,11.000,78,8.700,0.000,1.820,29.760,-95.370,29.750,-95.250,20.000
1,COAST,Houston,2019-01-01 01:00:00+00:00,11.100,9.600,80,7.800,0.000,1.810,29.760,-95.370,29.750,-95.250,20.000
2,COAST,Houston,2019-01-01 02:00:00+00:00,10.400,8.900,84,7.900,0.000,1.910,29.760,-95.370,29.750,-95.250,20.000
3,COAST,Houston,2019-01-01 03:00:00+00:00,9.000,7.400,92,7.700,0.000,1.820,29.760,-95.370,29.750,-95.250,20.000
4,COAST,Houston,2019-01-01 04:00:00+00:00,8.400,6.700,92,7.200,0.000,2.020,29.760,-95.370,29.750,-95.250,20.000


In [ ]:
# Validate the first downloaded frame and hourly continuity.
weather_value_columns = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "wind_speed_10m",
]

weather_time_differences = coast_2019_weather[
    "weather_time_utc"
].diff()

hourly_intervals_are_continuous = (
    weather_time_differences.dropna()
    .eq(pd.Timedelta(hours=1))
    .all()
)

weather_field_check = pd.DataFrame({
    "dtype": coast_2019_weather[
        weather_value_columns
    ].dtypes.astype(str),

    "missing_count": coast_2019_weather[
        weather_value_columns
    ].isna().sum(),

    "missing_rate": coast_2019_weather[
        weather_value_columns
    ].isna().mean(),

    "non_missing_count": coast_2019_weather[
        weather_value_columns
    ].notna().sum(),
})

weather_time_check = pd.Series({
    "row_count": len(coast_2019_weather),
    "utc_is_unique": coast_2019_weather[
        "weather_time_utc"
    ].is_unique,
    "utc_is_sorted": coast_2019_weather[
        "weather_time_utc"
    ].is_monotonic_increasing,
    "hourly_intervals_are_continuous": (
        hourly_intervals_are_continuous
    ),
}, name="value")

display(weather_field_check)

display(weather_time_check)


,dtype,missing_count,missing_rate,non_missing_count
temperature_2m,float64,0,0.000,8760
apparent_temperature,float64,0,0.000,8760
relative_humidity_2m,int64,0,0.000,8760
dew_point_2m,float64,0,0.000,8760
precipitation,float64,0,0.000,8760
wind_speed_10m,float64,0,0.000,8760


row_count                          8760
utc_is_unique                      True
utc_is_sorted                      True
hourly_intervals_are_continuous    True
Name: value, dtype: object

### 7. Download All Zones and Years

Retrieve or reuse yearly CSV files for all eight zones across 2019–2025. The 2025 batch extends to 2026-01-02 to cover the UTC alignment boundary; this extension is not a claim of 2026 load data.


In [ ]:
# Download or load cached annual weather files for all eight zones.
weather_frames = []

download_records = []

download_years = range(
    2019,
    2026,
)

for _, location_row in weather_locations.iterrows():
    current_zone = location_row["zone"]

    current_location_name = location_row["location_name"]

    current_latitude = location_row["latitude"]

    current_longitude = location_row["longitude"]

    for current_year in download_years:
        current_start_date = f"{current_year}-01-01"

        if current_year == 2025:
            current_end_date = "2026-01-02"
        else:
            current_end_date = f"{current_year}-12-31"

        raw_weather_file_path = (
            raw_weather_dir
            / f"{current_zone.lower()}_weather_{current_year}.csv"
        )

        if raw_weather_file_path.exists():
            print(
                f"Loading existing file: "
                f"{raw_weather_file_path.name}"
            )

            current_weather = pd.read_csv(
                raw_weather_file_path,
                parse_dates=["weather_time_utc"],
            )

            current_weather["weather_time_utc"] = pd.to_datetime(
                current_weather["weather_time_utc"],
                utc=True,
                errors="raise",
            )

            current_status = "loaded_existing"
        else:
            current_weather = download_zone_weather(
                zone=current_zone,
                location_name=current_location_name,
                latitude=current_latitude,
                longitude=current_longitude,
                start_date=current_start_date,
                end_date=current_end_date,
            )

            current_weather.to_csv(
                raw_weather_file_path,
                index=False,
            )

            current_status = "downloaded"

            time.sleep(1)

        weather_frames.append(
            current_weather
        )

        download_records.append({
            "zone": current_zone,
            "location_name": current_location_name,
            "download_year": current_year,
            "start_date": current_start_date,
            "end_date": current_end_date,
            "row_count": len(current_weather),
            "status": current_status,
            "file_path": str(raw_weather_file_path.relative_to(project_root)),
        })

        print(
            f"Completed {current_zone} {current_year}: "
            f"{len(current_weather):,} rows"
        )

all_weather_data = pd.concat(
    weather_frames,
    ignore_index=True,
)

all_weather_data = all_weather_data.sort_values(
    ["zone", "weather_time_utc"]
).reset_index(
    drop=True
)

weather_download_log = pd.DataFrame(
    download_records
)

print(
    "Combined weather shape:",
    all_weather_data.shape,
)

display(weather_download_log)


Completed COAST 2019: 8,760 rows
Completed COAST 2020: 8,784 rows
Completed COAST 2021: 8,760 rows
Completed COAST 2022: 8,760 rows
Completed COAST 2023: 8,760 rows
Completed COAST 2024: 8,784 rows
Completed COAST 2025: 8,808 rows
Completed EAST 2019: 8,760 rows
Completed EAST 2020: 8,784 rows
Completed EAST 2021: 8,760 rows
Completed EAST 2022: 8,760 rows
Completed EAST 2023: 8,760 rows
Completed EAST 2024: 8,784 rows
Completed EAST 2025: 8,808 rows
Completed FWEST 2019: 8,760 rows
Completed FWEST 2020: 8,784 rows
Completed FWEST 2021: 8,760 rows
Completed FWEST 2022: 8,760 rows
Completed FWEST 2023: 8,760 rows
Completed FWEST 2024: 8,784 rows
Completed FWEST 2025: 8,808 rows
Completed NORTH 2019: 8,760 rows
Completed NORTH 2020: 8,784 rows
Completed NORTH 2021: 8,760 rows
Completed NORTH 2022: 8,760 rows
Completed NORTH 2023: 8,760 rows
Completed NORTH 2024: 8,784 rows
Completed NORTH 2025: 8,808 rows
Completed NCENT 2019: 8,760 rows
Completed NCENT 2020: 8,784 rows
Completed NCENT 2

,zone,location_name,download_year,start_date,end_date,row_count,status,file_path
0,COAST,Houston,2019,2019-01-01,2019-12-31,8760,downloaded,data/...
1,COAST,Houston,2020,2020-01-01,2020-12-31,8784,downloaded,data/...
2,COAST,Houston,2021,2021-01-01,2021-12-31,8760,downloaded,data/...
3,COAST,Houston,2022,2022-01-01,2022-12-31,8760,downloaded,data/...
4,COAST,Houston,2023,2023-01-01,2023-12-31,8760,downloaded,data/...
5,COAST,Houston,2024,2024-01-01,2024-12-31,8784,downloaded,data/...
6,COAST,Houston,2025,2025-01-01,2026-01-02,8808,downloaded,data/...
7,EAST,Tyler,2019,2019-01-01,2019-12-31,8760,downloaded,data/...
8,EAST,Tyler,2020,2020-01-01,2020-12-31,8784,downloaded,data/...
9,EAST,Tyler,2021,2021-01-01,2021-12-31,8760,downloaded,data/...


### 8. Weather Download Log

Summarize the retrieval batches and save a log of the source files and row counts. Path entries use project-relative names.


In [ ]:
# Summarize and save the weather download log.
weather_download_log_path = (
    weather_data_dir
    / "weather_download_log.csv"
)

weather_download_log.to_csv(
    weather_download_log_path.relative_to(project_root),
    index=False,
)

download_summary_by_zone = (
    weather_download_log
    .groupby(
        "zone",
        as_index=False,
    )
    .agg(
        batch_count=(
            "download_year",
            "count",
        ),
        total_rows=(
            "row_count",
            "sum",
        ),
    )
)

display(download_summary_by_zone)

print(
    "Download log saved to:",
    weather_download_log_path.relative_to(project_root),
)


,zone,batch_count,total_rows
0,COAST,7,61416
1,EAST,7,61416
2,FWEST,7,61416
3,NCENT,7,61416
4,NORTH,7,61416
5,SCENT,7,61416
6,SOUTH,7,61416
7,WEST,7,61416


Download log saved to: data/weather/weather_download_log.csv


### 9. Combined Weather Data Validation

Check the expected set of zones, record counts, duplicate zone–hour keys, and the overall UTC span.


In [ ]:
# Validate the combined zone-level weather dataset.
expected_weather_zones = {
    "COAST",
    "EAST",
    "FWEST",
    "NORTH",
    "NCENT",
    "SOUTH",
    "SCENT",
    "WEST",
}

actual_weather_zones = set(
    all_weather_data["zone"].unique()
)

weather_rows_by_zone = (
    all_weather_data
    .groupby(
        "zone",
        as_index=False,
    )
    .agg(
        row_count=(
            "weather_time_utc",
            "size",
        ),
        utc_start=(
            "weather_time_utc",
            "min",
        ),
        utc_end=(
            "weather_time_utc",
            "max",
        ),
        location_count=(
            "location_name",
            "nunique",
        ),
    )
)

weather_structure_check = pd.Series({
    "total_row_count": len(all_weather_data),
    "zone_count": all_weather_data["zone"].nunique(),
    "zones_match_expected": (
        actual_weather_zones == expected_weather_zones
    ),
    "overall_utc_start": all_weather_data[
        "weather_time_utc"
    ].min(),
    "overall_utc_end": all_weather_data[
        "weather_time_utc"
    ].max(),
}, name="value")

display(weather_structure_check)

display(weather_rows_by_zone)


total_row_count                            491328
zone_count                                      8
zones_match_expected                         True
overall_utc_start       2019-01-01 00:00:00+00:00
overall_utc_end         2026-01-02 23:00:00+00:00
Name: value, dtype: object

,zone,row_count,utc_start,utc_end,location_count
0,COAST,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1
1,EAST,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1
2,FWEST,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1
3,NCENT,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1
4,NORTH,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1
5,SCENT,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1
6,SOUTH,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1
7,WEST,61416,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00,1


### 10. Missing-Value Validation

Count missing values for the six weather covariates by zone and across the combined dataset.


In [ ]:
# Count missing weather observations by zone.
weather_value_columns = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "dew_point_2m",
    "precipitation",
    "wind_speed_10m",
]

weather_missing_by_zone = (
    all_weather_data[
        ["zone", *weather_value_columns]
    ]
    .groupby("zone")
    .apply(
        lambda group: group[
            weather_value_columns
        ]
        .isna()
        .sum(),
        include_groups=False,
    )
)

total_weather_missing_values = int(
    weather_missing_by_zone.sum().sum()
)

display(weather_missing_by_zone)

print(
    "Total missing weather values:",
    total_weather_missing_values,
)


,temperature_2m,apparent_temperature,relative_humidity_2m,dew_point_2m,precipitation,wind_speed_10m
zone,,,,,,
COAST,0,0,0,0,0,0
EAST,0,0,0,0,0,0
FWEST,0,0,0,0,0,0
NCENT,0,0,0,0,0,0
NORTH,0,0,0,0,0,0
SCENT,0,0,0,0,0,0
SOUTH,0,0,0,0,0,0
WEST,0,0,0,0,0,0


Total missing weather values: 0


In [ ]:
# Check the continuity of zone–hour weather timestamps.
duplicate_zone_time_rows = int(
    all_weather_data.duplicated(
        subset=[
            "zone",
            "weather_time_utc",
        ],
        keep=False,
    ).sum()
)

zone_time_quality_records = []

for zone, zone_weather in all_weather_data.groupby(
    "zone"
):
    zone_weather = zone_weather.sort_values(
        "weather_time_utc"
    ).copy()

    utc_differences = zone_weather[
        "weather_time_utc"
    ].diff()

    non_hourly_intervals = utc_differences.loc[
        utc_differences.notna()
        & utc_differences.ne(
            pd.Timedelta(hours=1)
        )
    ]

    zone_time_quality_records.append({
        "zone": zone,
        "row_count": len(zone_weather),
        "utc_is_unique": zone_weather[
            "weather_time_utc"
        ].is_unique,
        "utc_is_sorted": zone_weather[
            "weather_time_utc"
        ].is_monotonic_increasing,
        "non_hourly_interval_count": len(
            non_hourly_intervals
        ),
        "utc_start": zone_weather[
            "weather_time_utc"
        ].min(),
        "utc_end": zone_weather[
            "weather_time_utc"
        ].max(),
    })

zone_time_quality = pd.DataFrame(
    zone_time_quality_records
)

display(zone_time_quality)

print(
    "Duplicate zone-time rows:",
    duplicate_zone_time_rows,
)


,zone,row_count,utc_is_unique,utc_is_sorted,non_hourly_interval_count,utc_start,utc_end
0,COAST,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00
1,EAST,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00
2,FWEST,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00
3,NCENT,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00
4,NORTH,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00
5,SCENT,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00
6,SOUTH,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00
7,WEST,61416,True,True,0,2019-01-01 00:00:00+00:00,2026-01-02 23:00:00+00:00


Duplicate zone-time rows: 0


### 11. Additional Quality Checks

Inspect time continuity and quality summaries before saving the processed weather table.


In [ ]:
# Build additional combined weather quality checks.
combined_weather_validation = pd.Series({
    "total_row_count": len(all_weather_data),
    "zone_count": all_weather_data["zone"].nunique(),
    "zones_match_expected": (
        actual_weather_zones == expected_weather_zones
    ),
    "all_zones_have_expected_rows": (
        weather_rows_by_zone["row_count"]
        .eq(61_416)
        .all()
    ),
    "total_missing_weather_values": (
        total_weather_missing_values
    ),
    "duplicate_zone_time_rows": (
        duplicate_zone_time_rows
    ),
    "all_zone_utc_values_unique": (
        zone_time_quality["utc_is_unique"].all()
    ),
    "all_zone_utc_values_sorted": (
        zone_time_quality["utc_is_sorted"].all()
    ),
    "total_non_hourly_intervals": int(
        zone_time_quality[
            "non_hourly_interval_count"
        ].sum()
    ),
}, name="value")

display(combined_weather_validation)


total_row_count                 491328
zone_count                           8
zones_match_expected              True
all_zones_have_expected_rows      True
total_missing_weather_values         0
duplicate_zone_time_rows             0
all_zone_utc_values_unique        True
all_zone_utc_values_sorted        True
total_non_hourly_intervals           0
Name: value, dtype: object

In [ ]:
# Data preparation or validation step.
combined_weather_validation


total_row_count                 491328
zone_count                           8
zones_match_expected              True
all_zones_have_expected_rows      True
total_missing_weather_values         0
duplicate_zone_time_rows             0
all_zone_utc_values_unique        True
all_zone_utc_values_sorted        True
total_non_hourly_intervals           0
Name: value, dtype: object

### 12. Processed Weather Outputs

Define output paths for cleaned weather data and validation reports.


In [ ]:
# Define processed data and quality-report destinations.
processed_weather_path = (
    processed_weather_dir
    / "ercot_zone_weather_2019_2026_boundary.parquet"
)

weather_validation_path = (
    processed_weather_dir
    / "weather_validation_summary.csv"
)

zone_time_quality_path = (
    processed_weather_dir
    / "zone_time_quality_summary.csv"
)

print("Processed weather:", processed_weather_path.relative_to(project_root))
print("Weather validation:", weather_validation_path.relative_to(project_root))
print("Zone-time quality:", zone_time_quality_path.relative_to(project_root))


Processed weather: data/weather/processed/ercot_zone_weather_2019_2026_boundary.parquet
Weather validation: data/weather/processed/weather_validation_summary.csv
Zone-time quality: data/weather/processed/zone_time_quality_summary.csv


In [ ]:
# Export the processed weather Parquet data and QA summaries.
all_weather_data.to_parquet(
    processed_weather_path.relative_to(project_root),
    index=False,
    engine="pyarrow",
)

weather_validation_output = (
    combined_weather_validation
    .rename_axis("metric")
    .reset_index()
)

weather_validation_output.to_csv(
    weather_validation_path.relative_to(project_root),
    index=False,
)

zone_time_quality.to_csv(
    zone_time_quality_path.relative_to(project_root),
    index=False,
)

print(
    "Processed weather saved to:",
    processed_weather_path.relative_to(project_root),
)

print(
    "Weather validation saved to:",
    weather_validation_path.relative_to(project_root),
)

print(
    "Zone-time quality saved to:",
    zone_time_quality_path.relative_to(project_root),
)


Processed weather saved to: data/weather/processed/ercot_zone_weather_2019_2026_boundary.parquet
Weather validation saved to: data/weather/processed/weather_validation_summary.csv
Zone-time quality saved to: data/weather/processed/zone_time_quality_summary.csv


### 12.1 Reload and Verify Saved Weather Data

Read the Parquet file and confirm its saved shape, UTC values, zone count, and missing/duplicate counts.


In [ ]:
# Reload the saved weather data and verify consistency.
reloaded_weather = pd.read_parquet(
    processed_weather_path.relative_to(project_root),
    engine="pyarrow",
)

weather_shape_matches = (
    reloaded_weather.shape == all_weather_data.shape
)

weather_utc_values_match = reloaded_weather[
    "weather_time_utc"
].equals(
    all_weather_data["weather_time_utc"]
)

reloaded_duplicate_zone_time_rows = int(
    reloaded_weather.duplicated(
        subset=[
            "zone",
            "weather_time_utc",
        ]
    ).sum()
)

reloaded_missing_weather_values = int(
    reloaded_weather[
        weather_value_columns
    ]
    .isna()
    .sum()
    .sum()
)

saved_weather_file_validation = pd.Series({
    "reloaded_row_count": len(reloaded_weather),
    "reloaded_column_count": len(
        reloaded_weather.columns
    ),
    "shape_matches": weather_shape_matches,
    "utc_values_match": weather_utc_values_match,
    "zone_count": reloaded_weather[
        "zone"
    ].nunique(),
    "duplicate_zone_time_rows": (
        reloaded_duplicate_zone_time_rows
    ),
    "missing_weather_values": (
        reloaded_missing_weather_values
    ),
}, name="value")

display(saved_weather_file_validation)


reloaded_row_count          491328
reloaded_column_count           14
shape_matches                 True
utc_values_match              True
zone_count                       8
duplicate_zone_time_rows         0
missing_weather_values           0
Name: value, dtype: object

In [ ]:
# Display the weather file validation summary.
saved_weather_file_validation


reloaded_row_count          491328
reloaded_column_count           14
shape_matches                 True
utc_values_match              True
zone_count                       8
duplicate_zone_time_rows         0
missing_weather_values           0
Name: value, dtype: object

## Relationship to Downstream Modeling

This stage builds the data foundation. ERA5 supplies historical reanalysis weather variables, but it is **not** the project's electricity-demand forecasting model. Subsequent analysis studies temperature–demand relationships, including cooling/heating degree days and regression. The forecasting stage then compares lag-based baselines with machine-learning models. This notebook focuses on data alignment and validation rather than fitting a demand model.


## 13. Load ERCOT Demand and Check UTC Coverage

Read the cleaned 2019–2025 load data, verify the expected zone fields, and confirm that available weather timestamps span the full load period. The load time range extends into early January 2026 when expressed in UTC.


In [ ]:
# Load cleaned ERCOT demand and confirm required columns.
processed_load_path = (
    project_root
    / "data"
    / "processed"
    / "ercot_hourly_load_2019_2025.parquet"
)

if not processed_load_path.exists():
    raise FileNotFoundError(
        f"Processed ERCOT load file not found: "
        f"{processed_load_path.relative_to(project_root)}"
    )

ercot_load = pd.read_parquet(
    processed_load_path.relative_to(project_root),
    engine="pyarrow",
)

zone_columns = [
    "COAST",
    "EAST",
    "FWEST",
    "NORTH",
    "NCENT",
    "SOUTH",
    "SCENT",
    "WEST",
]

required_load_columns = {
    "HourEnding_utc",
    "ERCOT",
    *zone_columns,
}

missing_load_columns = (
    required_load_columns
    - set(ercot_load.columns)
)

if missing_load_columns:
    raise ValueError(
        "Missing required load columns: "
        f"{sorted(missing_load_columns)}"
    )

ercot_load["HourEnding_utc"] = pd.to_datetime(
    ercot_load["HourEnding_utc"],
    utc=True,
    errors="raise",
)

print("ERCOT load file:", processed_load_path.relative_to(project_root))

print("ERCOT load shape:", ercot_load.shape)

display(
    ercot_load[
        [
            "HourEnding_utc",
            "ERCOT",
            *zone_columns,
        ]
    ].head(3)
)


ERCOT load file: data/processed/ercot_hourly_load_2019_2025.parquet
ERCOT load shape: (61368, 20)


,HourEnding_utc,ERCOT,COAST,EAST,FWEST,NORTH,NCENT,SOUTH,SCENT,WEST
0,2019-01-01 07:00:00+00:00,"37,081.443","9,783.595","1,264.194","3,164.721",827.759,"11,697.766","3,066.891","5,993.974","1,282.543"
1,2019-01-01 08:00:00+00:00,"37,258.990","9,726.062","1,270.494","3,178.951",830.256,"11,787.936","3,140.383","6,029.774","1,295.133"
2,2019-01-01 09:00:00+00:00,"37,300.188","9,654.374","1,269.504","3,200.881",831.506,"11,861.628","3,137.721","6,043.349","1,301.225"


In [ ]:
# Compare the UTC coverage of load and weather.
load_utc_start = ercot_load[
    "HourEnding_utc"
].min()

load_utc_end = ercot_load[
    "HourEnding_utc"
].max()

weather_utc_start = reloaded_weather[
    "weather_time_utc"
].min()

weather_utc_end = reloaded_weather[
    "weather_time_utc"
].max()

weather_covers_load_start = (
    weather_utc_start <= load_utc_start
)

weather_covers_load_end = (
    weather_utc_end >= load_utc_end
)

load_utc_is_unique = ercot_load[
    "HourEnding_utc"
].is_unique

load_utc_is_sorted = ercot_load[
    "HourEnding_utc"
].is_monotonic_increasing

load_weather_coverage_check = pd.Series({
    "load_row_count": len(ercot_load),
    "load_utc_start": load_utc_start,
    "load_utc_end": load_utc_end,
    "weather_utc_start": weather_utc_start,
    "weather_utc_end": weather_utc_end,
    "weather_covers_load_start": (
        weather_covers_load_start
    ),
    "weather_covers_load_end": (
        weather_covers_load_end
    ),
    "load_utc_is_unique": load_utc_is_unique,
    "load_utc_is_sorted": load_utc_is_sorted,
}, name="value")

display(load_weather_coverage_check)


load_row_count                                   61368
load_utc_start               2019-01-01 07:00:00+00:00
load_utc_end                 2026-01-01 06:00:00+00:00
weather_utc_start            2019-01-01 00:00:00+00:00
weather_utc_end              2026-01-02 23:00:00+00:00
weather_covers_load_start                         True
weather_covers_load_end                           True
load_utc_is_unique                                True
load_utc_is_sorted                                True
Name: value, dtype: object

In [ ]:
# Display UTC coverage diagnostics.
print(
    load_weather_coverage_check.to_string()
)


load_row_count                                   61368
load_utc_start               2019-01-01 07:00:00+00:00
load_utc_end                 2026-01-01 06:00:00+00:00
weather_utc_start            2019-01-01 00:00:00+00:00
weather_utc_end              2026-01-02 23:00:00+00:00
weather_covers_load_start                         True
weather_covers_load_end                           True
load_utc_is_unique                                True
load_utc_is_sorted                                True


In [ ]:
# Export the UTC coverage report.
load_weather_coverage_path = (
    processed_weather_dir
    / "load_weather_utc_coverage_check.csv"
)

load_weather_coverage_output = (
    load_weather_coverage_check
    .rename_axis("metric")
    .reset_index()
)

load_weather_coverage_output.to_csv(
    load_weather_coverage_path.relative_to(project_root),
    index=False,
)

print(
    "Load-weather UTC coverage report saved to:",
    load_weather_coverage_path.relative_to(project_root),
)


Load-weather UTC coverage report saved to: data/weather/processed/load_weather_utc_coverage_check.csv


## 14. Reshape Regional Load to Long Format

Convert eight zone-specific load columns to one row per `(HourEnding_utc, zone)` pair, preserving the system load and existing data-quality flag.


In [ ]:
# Reshape wide regional load observations into long format.
load_identifier_columns = [
    "HourEnding_utc",
    "ERCOT",
    "source_year",
    "zone_total_mismatch_flag",
]

missing_identifier_columns = (
    set(load_identifier_columns)
    - set(ercot_load.columns)
)

if missing_identifier_columns:
    raise ValueError(
        "Missing load identifier columns: "
        f"{sorted(missing_identifier_columns)}"
    )

regional_load_long = ercot_load.melt(
    id_vars=load_identifier_columns,
    value_vars=zone_columns,
    var_name="zone",
    value_name="load_mw",
)

regional_load_long = regional_load_long.sort_values(
    ["HourEnding_utc", "zone"]
).reset_index(
    drop=True
)

print(
    "Regional load long shape:",
    regional_load_long.shape,
)

display(
    regional_load_long.head(16)
)


Regional load long shape: (490944, 6)


,HourEnding_utc,ERCOT,source_year,zone_total_mismatch_flag,zone,load_mw
0,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,COAST,"9,783.595"
1,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,EAST,"1,264.194"
2,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,FWEST,"3,164.721"
3,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,NCENT,"11,697.766"
4,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,NORTH,827.759
5,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,SCENT,"5,993.974"
6,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,SOUTH,"3,066.891"
7,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,WEST,"1,282.543"
8,2019-01-01 08:00:00+00:00,"37,258.990",2019,False,COAST,"9,726.062"
9,2019-01-01 08:00:00+00:00,"37,258.990",2019,False,EAST,"1,270.494"


In [ ]:
# Validate the resulting zone–hour load records.
regional_load_duplicate_keys = int(
    regional_load_long.duplicated(
        subset=[
            "zone",
            "HourEnding_utc",
        ]
    ).sum()
)

regional_load_long_validation = pd.Series({
    "row_count": len(regional_load_long),
    "row_count_matches": (
        len(regional_load_long) == 490_944
    ),
    "zone_count": regional_load_long[
        "zone"
    ].nunique(),
    "duplicate_zone_time_keys": (
        regional_load_duplicate_keys
    ),
    "missing_load_values": int(
        regional_load_long[
            "load_mw"
        ].isna().sum()
    ),
    "nonpositive_load_values": int(
        (
            regional_load_long[
                "load_mw"
            ] <= 0
        ).sum()
    ),
    "flagged_long_rows": int(
        regional_load_long[
            "zone_total_mismatch_flag"
        ].sum()
    ),
}, name="value")

display(regional_load_long_validation)


row_count                   490944
row_count_matches             True
zone_count                       8
duplicate_zone_time_keys         0
missing_load_values              0
nonpositive_load_values          0
flagged_long_rows              384
Name: value, dtype: object

In [ ]:
# Display regional load validation metrics.
print(
    regional_load_long_validation.to_string()
)


row_count                   490944
row_count_matches             True
zone_count                       8
duplicate_zone_time_keys         0
missing_load_values              0
nonpositive_load_values          0
flagged_long_rows              384


## 15. Join Regional Load and Weather

Filter weather records to the load period and merge on both UTC hour and zone. A one-to-one validation check helps detect duplicates; unmatched observations are explicitly counted.


In [ ]:
# Restrict weather timestamps to the load interval.
weather_for_load_period = reloaded_weather.loc[
    reloaded_weather[
        "weather_time_utc"
    ].between(
        load_utc_start,
        load_utc_end,
        inclusive="both",
    )
].copy()

weather_for_load_period = weather_for_load_period.rename(
    columns={
        "weather_time_utc": "HourEnding_utc"
    }
)

weather_for_load_period = weather_for_load_period.sort_values(
    ["HourEnding_utc", "zone"]
).reset_index(
    drop=True
)

print(
    "Weather for load period shape:",
    weather_for_load_period.shape,
)

print(
    "Filtered weather UTC start:",
    weather_for_load_period["HourEnding_utc"].min(),
)

print(
    "Filtered weather UTC end:",
    weather_for_load_period["HourEnding_utc"].max(),
)


Weather for load period shape: (490944, 14)
Filtered weather UTC start: 2019-01-01 07:00:00+00:00
Filtered weather UTC end: 2026-01-01 06:00:00+00:00


In [ ]:
# Merge regional load and weather using zone and UTC hour.
load_weather_data = regional_load_long.merge(
    weather_for_load_period,
    on=[
        "HourEnding_utc",
        "zone",
    ],
    how="left",
    validate="one_to_one",
    indicator=True,
)

print(
    "Load-weather merged shape:",
    load_weather_data.shape,
)

print(
    load_weather_data[
        "_merge"
    ].value_counts(
        dropna=False
    )
)

display(
    load_weather_data.head(8)
)


Load-weather merged shape: (490944, 19)
_merge
both          490944
left_only          0
right_only         0
Name: count, dtype: int64


,HourEnding_utc,ERCOT,source_year,zone_total_mismatch_flag,zone,load_mw,location_name,temperature_2m,apparent_temperature,relative_humidity_2m,dew_point_2m,precipitation,wind_speed_10m,requested_latitude,requested_longitude,grid_latitude,grid_longitude,grid_elevation_m,_merge
0,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,COAST,"9,783.595",Houston,8.100,6.700,91,6.800,0.000,1.220,29.760,-95.370,29.750,-95.250,20.000,both
1,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,EAST,"1,264.194",Tyler,4.600,1.800,96,4.000,0.000,2.620,32.351,-95.301,32.250,-95.250,169.000,both
2,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,FWEST,"3,164.721",Midland,2.400,-3.600,75,-1.500,0.000,6.940,31.997,-102.078,32.000,-102.000,851.000,both
3,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,NCENT,"11,697.766",Dallas,5.900,2.300,83,3.200,0.000,3.920,32.777,-96.797,32.750,-96.750,137.000,both
4,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,NORTH,827.759,Wichita Falls,-0.500,-7.100,70,-5.300,0.000,7.100,33.914,-98.493,34.000,-98.500,291.000,both
5,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,SCENT,"5,993.974",San Antonio,4.800,2.600,90,3.300,0.000,1.210,29.424,-98.494,29.500,-98.500,203.000,both
6,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,SOUTH,"3,066.891",McAllen,9.400,7.400,69,4.100,0.000,1.250,26.203,-98.230,26.250,-98.250,39.000,both
7,2019-01-01 07:00:00+00:00,"37,081.443",2019,False,WEST,"1,282.543",Abilene,0.800,-5.200,78,-2.600,0.000,6.530,32.449,-99.733,32.500,-99.750,525.000,both


## 16. Validate and Export the Modeling Dataset

Check coverage, missing weather values, duplicate zone–hour records, and row counts, then save the finalized joined dataset and QA summary for subsequent modeling.


In [ ]:
# Evaluate final join coverage and QA metrics.
matched_load_weather_rows = int(
    load_weather_data[
        "_merge"
    ].eq(
        "both"
    ).sum()
)

unmatched_load_rows = int(
    load_weather_data[
        "_merge"
    ].eq(
        "left_only"
    ).sum()
)

merged_weather_missing_values = int(
    load_weather_data[
        weather_value_columns
    ]
    .isna()
    .sum()
    .sum()
)

merged_duplicate_zone_time_keys = int(
    load_weather_data.duplicated(
        subset=[
            "zone",
            "HourEnding_utc",
        ]
    ).sum()
)

load_weather_validation = pd.Series({
    "row_count": len(load_weather_data),
    "row_count_matches": (
        len(load_weather_data) == 490_944
    ),
    "matched_rows": matched_load_weather_rows,
    "unmatched_load_rows": unmatched_load_rows,
    "weather_match_rate": (
        matched_load_weather_rows
        / len(load_weather_data)
    ),
    "missing_weather_values": (
        merged_weather_missing_values
    ),
    "duplicate_zone_time_keys": (
        merged_duplicate_zone_time_keys
    ),
    "zone_count": load_weather_data[
        "zone"
    ].nunique(),
    "missing_load_values": int(
        load_weather_data[
            "load_mw"
        ].isna().sum()
    ),
    "flagged_long_rows": int(
        load_weather_data[
            "zone_total_mismatch_flag"
        ].sum()
    ),
}, name="value")

print(
    load_weather_validation.to_string()
)


row_count                   490944
row_count_matches             True
matched_rows                490944
unmatched_load_rows              0
weather_match_rate           1.000
missing_weather_values           0
duplicate_zone_time_keys         0
zone_count                       8
missing_load_values              0
flagged_long_rows              384


In [ ]:
# Remove the temporary merge indicator and order the rows.
final_load_weather_data = load_weather_data.drop(
    columns=[
        "_merge"
    ]
).copy()

final_load_weather_data = final_load_weather_data.sort_values(
    [
        "HourEnding_utc",
        "zone",
    ]
).reset_index(
    drop=True
)

print(
    "Final load-weather shape:",
    final_load_weather_data.shape,
)

print(
    "Final columns:",
    final_load_weather_data.columns.tolist(),
)


Final load-weather shape: (490944, 18)
Final columns: ['HourEnding_utc', 'ERCOT', 'source_year', 'zone_total_mismatch_flag', 'zone', 'load_mw', 'location_name', 'temperature_2m', 'apparent_temperature', 'relative_humidity_2m', 'dew_point_2m', 'precipitation', 'wind_speed_10m', 'requested_latitude', 'requested_longitude', 'grid_latitude', 'grid_longitude', 'grid_elevation_m']


In [ ]:
# Export the joined modeling dataset and QA report.
modeling_data_dir = (
    project_root
    / "data"
    / "modeling"
)

modeling_data_dir.mkdir(
    parents=True,
    exist_ok=True,
)

final_load_weather_path = (
    modeling_data_dir
    / "regional_load_weather_2019_2025.parquet"
)

load_weather_validation_path = (
    modeling_data_dir
    / "regional_load_weather_validation.csv"
)

final_load_weather_data.to_parquet(
    final_load_weather_path.relative_to(project_root),
    index=False,
    engine="pyarrow",
)

load_weather_validation_output = (
    load_weather_validation
    .rename_axis("metric")
    .reset_index()
)

load_weather_validation_output.to_csv(
    load_weather_validation_path.relative_to(project_root),
    index=False,
)

print(
    "Final load-weather data saved to:",
    final_load_weather_path.relative_to(project_root),
)

print(
    "Load-weather validation saved to:",
    load_weather_validation_path.relative_to(project_root),
)


Final load-weather data saved to: data/modeling/regional_load_weather_2019_2025.parquet
Load-weather validation saved to: data/modeling/regional_load_weather_validation.csv


In [ ]:
# Reload the joined dataset and verify the saved contents.
reloaded_load_weather = pd.read_parquet(
    final_load_weather_path.relative_to(project_root),
    engine="pyarrow",
)

final_file_validation = pd.Series({
    "reloaded_row_count": len(
        reloaded_load_weather
    ),
    "reloaded_column_count": len(
        reloaded_load_weather.columns
    ),
    "shape_matches": (
        reloaded_load_weather.shape
        == final_load_weather_data.shape
    ),
    "zone_count": reloaded_load_weather[
        "zone"
    ].nunique(),
    "utc_values_match": (
        reloaded_load_weather[
            "HourEnding_utc"
        ].equals(
            final_load_weather_data[
                "HourEnding_utc"
            ]
        )
    ),
    "missing_load_values": int(
        reloaded_load_weather[
            "load_mw"
        ].isna().sum()
    ),
    "missing_weather_values": int(
        reloaded_load_weather[
            weather_value_columns
        ].isna().sum().sum()
    ),
}, name="value")

print(
    final_file_validation.to_string()
)


reloaded_row_count        490944
reloaded_column_count         18
shape_matches               True
zone_count                     8
utc_values_match            True
missing_load_values            0
missing_weather_values         0
